# Conflict Projection — paper experiment pipeline

This notebook follows the thesis results order: **RAMDocs single-answer**, **ConflictBank 1-vs-3**, then **QACC natural conflict**, followed by ablations and diagnostics. Reusable logic is imported from `src/conflict_projection`; this notebook only fixes cohorts/configurations, runs methods, and reports paired results.

## 0. Project bootstrap and all imports

Keep this as the first code cell. It makes the src-layout package importable even without an editable install.

In [ ]:
from pathlib import Path
import getpass
import os
import sys
import numpy as np
import pandas as pd

project_hint = Path(os.environ.get(
    "CONFLICT_PROJECTION_ROOT",
    "/Users/marthalee/Documents/heidelberg/第四学期/Knowledge Conflicts in LLM/conflict_projection",
)).expanduser().resolve()
search_starts = [Path.cwd().resolve(), project_hint]
project_root = next((
    candidate for start in search_starts for candidate in (start, *start.parents)
    if (candidate / "pyproject.toml").is_file()
), None)
if project_root is None:
    raise FileNotFoundError("Set CONFLICT_PROJECTION_ROOT to the project directory.")
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from conflict_projection.config import ModelConfig, ProjectPaths, ProjectionConfig
from conflict_projection.datasets import (
    describe_instances, download_qacc, load_conflictbank, load_qacc, load_ramdocs,
)
from conflict_projection.diagnostics import metadata_coverage, projection_diagnostics
from conflict_projection.evaluation import (
    exact_mcnemar, paired_delta_interval, result_row, run_evaluation, subset_result,
)
from conflict_projection.llm import OpenAIChatClient, SQLitePromptCache
from conflict_projection.methods import (
    make_concat_method, make_madam_method, make_projected_method,
)
from conflict_projection.runtime import ModelRuntime
from conflict_projection.scoring import (
    parse_option, present_option, present_qacc, present_strict,
)

paths = ProjectPaths(project_root)
paths

## 1. Run mode, credentials, and shared model settings

Use `smoke` while checking the pipeline. Change to `paper` only when ready for the full API cost. All paper rows use `gpt-4o-mini`, temperature 0, and the same response cache.

In [ ]:
RUN_MODE = "smoke"  # "smoke" or "paper"
QACC_COHORT = "official_test"  # recommended; see the cohort audit below
CONFLICTBANK_PROTOCOL = "unsupervised"  # or "legacy_label_cluster" for audit only
RUN_ABLATIONS = RUN_MODE == "paper"
RUN_DIAGNOSTICS = False

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

model_config = ModelConfig(
    chat_model="gpt-4o-mini", temperature=0.0, max_tokens=400, seed=42,
)
client = OpenAIChatClient(model_config, cache=SQLitePromptCache(paths.cache_db))
runtime = ModelRuntime(model_config)
print(
    f"mode={RUN_MODE}, qacc_cohort={QACC_COHORT}, "
    f"conflictbank_protocol={CONFLICTBANK_PROTOCOL}, model={model_config.chat_model}"
)

## 2. Load data and freeze evaluation cohorts

The same materialized instance list is passed to every method in a benchmark. ConflictBank reuses `../inter/conflictbank_features.parquet` when available, preserving the existing 1000-qid order and precomputed g2/g6 values.

In [ ]:
ramdocs_all = load_ramdocs(split="test")
ramdocs_single_all = [item for item in ramdocs_all if len(item.gold_answers) == 1]

legacy_cb_features = project_root.parent / "inter/conflictbank_features.parquet"
conflictbank_all = load_conflictbank(
    n=1000, seed=42,
    precomputed_features_path=legacy_cb_features if legacy_cb_features.exists() else None,
)

qacc_path = download_qacc(paths.data, revision="main")
qacc_official_test = load_qacc(qacc_path, split="test")
qacc_legacy_first_813 = load_qacc(qacc_path, split=None)[:813]

if RUN_MODE == "paper":
    ramdocs_eval = ramdocs_single_all[:100]
    conflictbank_eval = conflictbank_all[:1000]
    qacc_eval = (
        qacc_official_test if QACC_COHORT == "official_test" else qacc_legacy_first_813
    )
else:
    ramdocs_eval = ramdocs_single_all[:5]
    conflictbank_eval = conflictbank_all[:5]
    qacc_source = qacc_official_test if QACC_COHORT == "official_test" else qacc_legacy_first_813
    qacc_eval = [item for item in qacc_source if item.has_conflict][:5] + [
        item for item in qacc_source if not item.has_conflict
    ][:5]

print("RAMDocs single-answer:", describe_instances(ramdocs_eval))
print("ConflictBank 1-vs-3:", describe_instances(conflictbank_eval))
print("QACC selected cohort:", describe_instances(qacc_eval))

### QACC cohort audit — resolve this before final reporting

The official `test` split contains 813 items but **207 conflict / 606 non-conflict**. The thesis draft's **166 / 647** is reproduced by taking the first 813 rows of the full 1617-row file; that cohort mixes train/dev/test. Keep `official_test` for a defensible evaluation, or use `legacy_first_813` only to reproduce the existing draft and relabel it accurately.

In [ ]:
cohort_audit = pd.DataFrame([
    {"cohort": "official_test", **describe_instances(qacc_official_test)},
    {"cohort": "legacy_first_813", **describe_instances(qacc_legacy_first_813)},
])
display(cohort_audit[["cohort", "instances", "conflicting_instances", "documents"]])
if RUN_MODE == "paper":
    assert len(ramdocs_eval) == 100
    assert len(conflictbank_eval) == 1000
    assert len(qacc_eval) == 813

## 3. Part 1A — RAMDocs single-answer (`n=100`)

Rows: concat baseline, MADAM-RAG, and MaxEnt top-5. All three consume the identical frozen slice.

In [ ]:
ram_projection = ProjectionConfig(
    features=("g2", "h", "g6"), top_k=5,
    threshold_strategy="legacy_mean", use_retriever_prior=False,
)
ram_methods = {
    "Baseline (concat)": make_concat_method(client, "ramdocs"),
    "MADAM-RAG": make_madam_method(client, "ramdocs", rounds=3),
    "MaxEnt (ours)": make_projected_method(
        client, runtime, "ramdocs", ram_projection, annotate_weights=True,
    ),
}
ram_results = {
    name: run_evaluation(
        ramdocs_eval, method, name, matcher=present_strict,
        output_path=paths.outputs / f"{RUN_MODE}-ramdocs-{name.lower().replace(' ', '-')}.jsonl",
    )
    for name, method in ram_methods.items()
}

In [ ]:
ram_table = pd.DataFrame([result_row(result) for result in ram_results.values()])
display(ram_table)
for comparator in ["Baseline (concat)", "MADAM-RAG"]:
    print("MaxEnt vs", comparator)
    print("  paired delta/CI:", paired_delta_interval(
        ram_results["MaxEnt (ours)"], ram_results[comparator]
    ))
    print("  McNemar:", exact_mcnemar(
        ram_results["MaxEnt (ours)"], ram_results[comparator]
    ))

## 4. Part 1B — ConflictBank 1-vs-3 (`n=1000`)

Rows: concat baseline and MaxEnt. OAR is implemented as exact accuracy of the returned A–D option. The recommended protocol clusters evidence without labels. The old `legacy_label_cluster` protocol uses the dataset's default/conflict category to protect the known-correct default document; it is retained only to audit earlier numbers and should not be reported as an unlabeled method.

In [ ]:
if CONFLICTBANK_PROTOCOL == "legacy_label_cluster":
    conflictbank_projection = ProjectionConfig(
        features=("g2", "h", "g6"), top_k=None,
        deduplicate_documents=False, cluster_strategy="metadata",
        cluster_floors=(0.45, 0.15), threshold_strategy="legacy_mean",
    )
elif CONFLICTBANK_PROTOCOL == "unsupervised":
    conflictbank_projection = ProjectionConfig(
        features=("g2", "h", "g6"), top_k=None,
        deduplicate_documents=False, cluster_strategy="kmeans",
        cluster_floor=0.05, threshold_strategy="legacy_mean",
    )
else:
    raise ValueError(f"Unknown ConflictBank protocol: {CONFLICTBANK_PROTOCOL}")
conflictbank_methods = {
    "Baseline (concat)": make_concat_method(client, "conflictbank"),
    "MaxEnt (ours)": make_projected_method(
        client, runtime, "conflictbank", conflictbank_projection,
        annotate_weights=True,
    ),
}
conflictbank_results = {
    name: run_evaluation(
        conflictbank_eval, method, name, parser=parse_option, matcher=present_option,
        output_path=paths.outputs / f"{RUN_MODE}-conflictbank-{name.lower().replace(' ', '-')}.jsonl",
    )
    for name, method in conflictbank_methods.items()
}

In [ ]:
conflictbank_table = pd.DataFrame([
    {
        **result_row(result), "metric": "OAR / accuracy",
        "protocol": CONFLICTBANK_PROTOCOL,
    }
    for result in conflictbank_results.values()
])
display(conflictbank_table)
print("paired delta/CI:", paired_delta_interval(
    conflictbank_results["MaxEnt (ours)"], conflictbank_results["Baseline (concat)"]
))
print("McNemar:", exact_mcnemar(
    conflictbank_results["MaxEnt (ours)"], conflictbank_results["Baseline (concat)"]
))

## 5. Part 2 — QACC natural conflict (`n=813`)

Both methods pass **all snippets**. MaxEnt does not truncate or deduplicate; it reorders all snippets and annotates relative evidence weights. `g4` uses provenance internally. Source/date strings are not separately exposed, avoiding an additional prompt-level intervention.

In [ ]:
qacc_projection = ProjectionConfig(
    features=("g2", "h", "g6", "g4"), top_k=None,
    deduplicate_documents=False, threshold_strategy="legacy_mean",
)
qacc_methods = {
    "Baseline (concat)": make_concat_method(client, "qacc"),
    "MaxEnt (ours)": make_projected_method(
        client, runtime, "qacc", qacc_projection,
        annotate_weights=True, expose_metadata=False,
    ),
}
qacc_results = {
    name: run_evaluation(
        qacc_eval, method, name, matcher=present_qacc,
        output_path=paths.outputs / f"{RUN_MODE}-qacc-{QACC_COHORT}-{name.lower().replace(' ', '-')}.jsonl",
    )
    for name, method in qacc_methods.items()
}

In [ ]:
qacc_conflict_ids = [item.instance_id for item in qacc_eval if item.has_conflict]
qacc_nonconflict_ids = [item.instance_id for item in qacc_eval if not item.has_conflict]
qacc_subsets = {}
for method_name, result in qacc_results.items():
    qacc_subsets[(method_name, "EM-C")] = subset_result(
        result, qacc_conflict_ids, name=f"{method_name} / EM-C"
    )
    qacc_subsets[(method_name, "EM-NC")] = subset_result(
        result, qacc_nonconflict_ids, name=f"{method_name} / EM-NC"
    )
    qacc_subsets[(method_name, "EM-T")] = result

qacc_rows = []
for method_name in qacc_methods:
    row = {"method": method_name}
    for metric in ["EM-C", "EM-NC", "EM-T"]:
        result = qacc_subsets[(method_name, metric)]
        stats = result_row(result)
        row[metric] = stats["exact"]
        row[f"{metric} 95% CI"] = (stats["exact_ci_low"], stats["exact_ci_high"])
    qacc_rows.append(row)
qacc_table = pd.DataFrame(qacc_rows)
display(qacc_table)

In [ ]:
for metric in ["EM-C", "EM-NC", "EM-T"]:
    ours = qacc_subsets[("MaxEnt (ours)", metric)]
    baseline = qacc_subsets[("Baseline (concat)", metric)]
    print(metric)
    print("  paired delta/CI:", paired_delta_interval(ours, baseline))
    print("  McNemar:", exact_mcnemar(ours, baseline))

## 6. QACC ablations required by the text/appendix

These rows isolate (a) top-5 filtering versus all-snippet annotation and (b) the contribution of provenance feature `g4`. They are disabled in smoke mode to avoid accidental API cost.

In [ ]:
qacc_ablation_results = {}
if RUN_ABLATIONS:
    ablation_configs = {
        "MaxEnt top-5": ProjectionConfig(
            features=("g2", "h", "g6", "g4"), top_k=5,
            deduplicate_documents=False, threshold_strategy="legacy_mean",
        ),
        "MaxEnt all, no g4": ProjectionConfig(
            features=("g2", "h", "g6"), top_k=None,
            deduplicate_documents=False, threshold_strategy="legacy_mean",
        ),
    }
    for name, config in ablation_configs.items():
        method = make_projected_method(
            client, runtime, "qacc", config, annotate_weights=True, expose_metadata=False,
        )
        qacc_ablation_results[name] = run_evaluation(
            qacc_eval, method, name, matcher=present_qacc,
            output_path=paths.outputs / f"paper-qacc-{QACC_COHORT}-{name.lower().replace(' ', '-')}.jsonl",
        )
    display(pd.DataFrame([result_row(result) for result in qacc_ablation_results.values()]))
else:
    print("Ablations skipped; set RUN_ABLATIONS=True when ready.")

## 7. Projection diagnostics (no LLM calls)

Inspect convergence, constraint violations, KL movement, lambdas, and source-tier weights before trusting a full run.

In [ ]:
print("QACC metadata coverage:", metadata_coverage(qacc_eval))
if RUN_DIAGNOSTICS:
    qacc_diagnostics = projection_diagnostics(
        qacc_eval, runtime, qacc_projection, n=min(100, len(qacc_eval))
    )
    display(qacc_diagnostics)
else:
    print("Projection diagnostics skipped; set RUN_DIAGNOSTICS=True to run.")

## 8. Export paper-facing result tables

CSV files are derived from the current run rather than hand-entered. Before copying numbers into LaTeX, confirm `RUN_MODE`, `QACC_COHORT`, dataset revisions, and output JSONL files.

In [ ]:
paper_main_table = pd.concat([
    ram_table.assign(benchmark="RAMDocs single-answer", metric="EM"),
    conflictbank_table.assign(benchmark="ConflictBank 1-vs-3"),
], ignore_index=True)
paper_main_table.to_csv(paths.outputs / f"{RUN_MODE}-table-main.csv", index=False)
qacc_table.to_csv(paths.outputs / f"{RUN_MODE}-table-qacc-{QACC_COHORT}.csv", index=False)
display(paper_main_table)
display(qacc_table)